# Sprint Block E+F+G — Geocell prediction (exploratory; superseded)

> **Status.** This notebook predicts a **geocell** (which ~55–880 km grid square of the
> Earth a sentence describes). That is *not* the thesis task. The thesis task — ranking the
> correct **OSM tag** for a paraphrased place, evaluated against Yousaf, Schwartz & Wolter
> (2023) — is in **Sprint Block H** (base-paper reproduction + adjective extension) and
> **Sprint Block I** (guided FP-Growth + Bayesian Elastic Net). The fusion of mined adjective
> patterns did **not** beat the text-only baseline on the geocell task, because a transformer
> already reads the adjectives from raw text; the contribution only shows on the tag task.
> This notebook is kept as the record of that exploration.

**Runtime: CPU for Block E, T4 GPU for Blocks F+G.**

| Block | Stage | Produces |
|-------|-------|----------|
| E1 | Geocell labelling + place-based splits | `y` labels, train/dev/test indices |
| E2 | Guided FP-Growth | frequent (adjective, noun) itemsets |
| E3 | Elastic-net pattern weighting | per-pattern geo-signal coefficients |
| F1 | DistilBERT baseline (text only) | reference model |
| F2 | DistilBERT fusion (text + mined patterns) | fusion model |
| G1 | Internal eval: Hits@k + nDCG@20 + ablation | geocell results table |
| G2 | External eval on the authors' dataset | (superseded by Block H) |

**Prerequisites:** `annotated/annotated_v1.parquet` and `transactions/transactions_v1.txt`
from Block C+D.


## Cell 1 — Mount, config, installs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob
ROOT = '/content/drive/MyDrive/geoproject'
os.environ['HF_HOME'] = f'{ROOT}/models_cache/hf'   # HF models cached on Drive
os.makedirs(f'{ROOT}/models', exist_ok=True)
os.makedirs(f'{ROOT}/results', exist_ok=True)

In [ ]:
%%capture
!pip install -q mlxtend scikit-learn scipy "transformers>=4.40" "datasets>=2.19" accelerate

In [ ]:
import pandas as pd, numpy as np

annot = pd.read_parquet(f'{ROOT}/annotated/annotated_v1.parquet')
# Records without coordinates cannot supervise a geotagger — drop them here,
# not earlier, so they remain available for unsupervised pattern mining.
annot = annot.dropna(subset=['lat', 'lon']).reset_index(drop=True)
print(f'Anchored, annotated sentences: {len(annot)}')
print(annot.source.value_counts())

## Block E1 / Cell 2 — Geocell labels + place-based splits

**Label design.** DistilBERT predicts a *geocell* (discretized region), not raw
coordinates — classification over cells is far more learnable at ~30–50k examples
than coordinate regression, and it directly supports Hits@k evaluation (rank cells
by softmax probability).

We use a simple two-level adaptive grid instead of H3/S2 to keep dependencies
minimal and the scheme fully transparent in the paper:
1. Quantize to fine cells of `FINE_DEG` degrees (~55 km at 0.5°).
2. Any fine cell with fewer than `MIN_CELL` training examples merges into its
   coarse parent (`COARSE_DEG` degrees). This prevents a long tail of 1-example
   classes that would make the softmax unlearnable.

**Split design — the non-negotiable.** We split by *place*, not by sentence.
Multiple sentences describe the same landmark (a Wikivoyage page's listings, its
prose, a Commons caption of the same fort). Sentence-level splits would leak
near-identical descriptions across train/test and inflate every metric. The group
key is `place_name` when present, else the fine geocell — so unnamed records
from the same spot also stay on one side of the split.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# --- MULTI-LEVEL ADAPTIVE GEOCELL GRID --------------------------------------
# Earlier two-level scheme (0.5 deg fine -> 2 deg coarse) bottomed out at ~3,200
# classes on this globally-spread corpus: there are ~3k populated 2-deg cells
# and MIN_CELL alone cannot push below that floor. A hierarchical ladder fixes
# it -- this is the standard adaptive-geocell construction (Weyand et al. 2016,
# "PlaNet"): geocell = the FINEST ladder level at which this location's cell
# holds >= MIN_CELL records; walk coarser until satisfied; coarsest level is the
# catch-all. MIN_CELL now directly controls both granularity and class count
# (raise it -> fewer, coarser classes). Target: 300-800 classes so sampling
# variance does not swamp the max|coef| geo-signal and the saga fit stays sane.
# MIN_CELL 800 -> 1500 lands ~600 classes on this corpus (800 gave 1,132).
GRID_LADDER = [0.5, 1.0, 2.0, 4.0, 8.0]   # degrees, fine -> coarse
MIN_CELL    = 1500

def cell_id(lat, lon, deg):
    """Quantize a coordinate to a grid cell string, e.g. 'g0.5_67_146'.
    Offsetting by 90/180 keeps indices non-negative for readability."""
    return f'g{deg}_{int((lat + 90) // deg)}_{int((lon + 180) // deg)}'

for deg in GRID_LADDER:
    annot[f'grid_{deg}'] = [cell_id(a, o, deg) for a, o in zip(annot.lat, annot.lon)]

geocell = pd.Series(index=annot.index, dtype=object)
remaining = annot.index
for deg in GRID_LADDER:
    counts = annot.loc[remaining, f'grid_{deg}'].value_counts()
    big = set(counts[counts >= MIN_CELL].index)
    hit = annot.loc[remaining, f'grid_{deg}'].isin(big)
    geocell.loc[remaining[hit]] = annot.loc[remaining[hit], f'grid_{deg}']
    remaining = remaining[~hit]
    print(f'  ladder {deg:>4} deg: {int(hit.sum()):>7,} records placed, '
          f'{len(remaining):>7,} still coarser')
geocell.loc[remaining] = annot.loc[remaining, f'grid_{GRID_LADDER[-1]}']
annot['geocell'] = geocell

# Group key for leakage-safe splitting: place_name when present, else finest cell
annot['group'] = annot.place_name.fillna('')
annot.loc[annot.group == '', 'group'] = annot['grid_0.5']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, holdout_idx = next(gss.split(annot, groups=annot.group))
hold = annot.iloc[holdout_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
dev_rel, test_rel = next(gss2.split(hold, groups=hold.group))

annot['split'] = 'train'
annot.iloc[holdout_idx[dev_rel],  annot.columns.get_loc('split')] = 'dev'
annot.iloc[holdout_idx[test_rel], annot.columns.get_loc('split')] = 'test'

# Freeze the label space to classes seen in training; unseen-at-train cells in
# dev/test map to a catch-all OTHER class (reported honestly as always-wrong).
train_cells = set(annot.loc[annot.split == 'train', 'geocell'])
annot['label_cell'] = np.where(annot.geocell.isin(train_cells), annot.geocell, 'OTHER')
classes = sorted(annot.label_cell.unique())
cls2id = {c: i for i, c in enumerate(classes)}
annot['label'] = annot.label_cell.map(cls2id)

# --- SCALE GUARD -------------------------------------------------------
# saga's cost scales with samples x features x CLASSES. Keep the label space
# small enough that E3 finishes in a session and the DistilBERT head is sane.
_n_cls = len(classes)
if _n_cls > 1500:
    print(f'!! WARNING: {_n_cls} geocell classes is very large.')
    print(f'   Raise MIN_CELL (currently {MIN_CELL}) and re-run this cell.')
elif _n_cls > 800:
    print(f'!  NOTE: {_n_cls} geocell classes -- workable but above the 300-800')
    print(f'   target; nudge MIN_CELL up if E3 is slow.')
else:
    print(f'OK: {_n_cls} geocell classes (in target range).')
# -----------------------------------------------------------------------

print(f'Geocell classes: {len(classes)} | split sizes:')
print(annot.split.value_counts())
print(f"Leakage check — groups shared between train and test: "
      f"{len(set(annot[annot.split=='train'].group) & set(annot[annot.split=='test'].group))} (must be 0)")
annot.to_parquet(f'{ROOT}/annotated/annotated_v1_labeled.parquet', index=False)

## Block E2 / Cell 3 — Guided FP-Growth

**What "guided" means here.** Plain FP-Growth would happily mine itemsets of pure
context nouns. We guide it two ways: (1) transactions were *constructed* from
typed items (`amod:`, `compound:`, `type:`, `ctx:`) in Block D, and (2) after
mining we keep only itemsets containing at least one `amod:` item — the
adjective-bearing patterns that are this project's contribution.

**Support threshold.** Mined on the TRAIN split only (patterns are features;
learning them from test would leak). `MIN_SUPPORT = 0.0001` on ~798k training
transactions ≈ patterns appearing ≥ 80 times — frequent enough for stable
elastic-net coefficients, rare enough to keep discriminative regional patterns.
(The value and counts printed by the next code cell are authoritative.)


In [ ]:
import os, json, datetime

p = f'{ROOT}/results/frequent_itemsets.json'
print(f'Exact size: {os.path.getsize(p):,} bytes')
print(f'Written at: {datetime.datetime.fromtimestamp(os.path.getmtime(p))}')

raw = open(p).read()
print(f'Char length: {len(raw):,}')
print(f'First 300 chars: {raw[:300]}')

data = json.loads(raw)
print(f'\nRecords: {len(data)}')
if data:
    print('Sample record:', data[0])

In [ ]:
import time
from collections import Counter

# MIN_SUPPORT lowered 0.002 -> 0.0001 (thesis config). At 0.002 the only
# surviving patterns are corpus-wide boilerplate ('good place', 'small town')
# with no geographic signal -- that silently invalidates the adjectives-help
# test. 0.0001 (~80 occurrences over ~798k train transactions) reaches the
# discriminative regional tail the elastic net is meant to weigh.
MIN_SUPPORT = 0.0001

_t0 = time.time()
def _lap(msg): print(f'[{time.time() - _t0:6.1f}s] {msg}', flush=True)

_tx = [json.loads(t) for t in annot.loc[annot.split == 'train', 'transaction']]
tx_sets = [frozenset(t) for t in _tx if t]
del _tx
N = len(tx_sets)
MIN_COUNT = MIN_SUPPORT * N
print(f'Training transactions: {N:,}   (support {MIN_SUPPORT} = >= {MIN_COUNT:.0f} occurrences)')

# --- GUIDED MINING, COMPUTED DIRECTLY --------------------------------------
# The guided constraint keeps ONLY itemsets containing >= 1 amod: item, and
# Block E3 reads back only the amod:/type: items. Running full FP-Growth over
# all ~7k items and then filtering therefore (a) wastes time/RAM materialising
# itemsets that are immediately discarded, and (b) OOMs the runtime -- mlxtend's
# dense fpgrowth at this vocabulary size holds the whole FP-tree + a frozenset
# result table in memory.
#
# Counting the adjective-anchored itemsets directly yields the IDENTICAL guided
# output: same itemsets, same supports (count / N, unchanged N), in seconds and
# flat memory. Methods-section wording: "guided frequent-itemset mining,
# restricted a priori to the adjective-anchored sublattice (supports identical
# to unconstrained FP-Growth followed by the amod filter)."
#
# We mine 1- and 2-itemsets. 3-itemsets are omitted deliberately: they add no
# new amod:/type: item to the E3 feature vocabulary (every such item already
# appears in a shorter itemset), so their only use is a co-occurrence figure,
# and enumerating amod-anchored triples over a 7k vocabulary is the exact
# combinatorial blow-up we are avoiding. Re-add as a targeted pass if a triple
# figure is needed.
# ponytail: direct count instead of FP-Growth; add an Apriori-join triple pass
#           only if the paper needs a 3-itemset co-occurrence figure.

# 1-itemsets over ALL item types (needed so a pair's partner support is known)
c1 = Counter()
for s in tx_sets:
    c1.update(s)
freq1 = {i for i, k in c1.items() if k >= MIN_COUNT}
amods = {i for i in freq1 if i.startswith('amod:')}
_lap(f'1-itemsets: {len(freq1):,} frequent of {len(c1):,}  ({len(amods):,} are amod:)')

# 2-itemsets with >= 1 amod: enumerate only pairs anchored on a frequent amod
# whose partner is itself frequent (Apriori prune -> the Counter stays bounded).
c2 = Counter()
for s in tx_sets:
    sa = amods.intersection(s)
    if not sa:
        continue
    partners = freq1.intersection(s)
    for a in sa:
        for o in partners:
            if o != a:
                c2[frozenset((a, o))] += 1
freq2 = {p: k for p, k in c2.items() if k >= MIN_COUNT}
del c2
_lap(f'2-itemsets (amod-anchored): {len(freq2):,} frequent')

# Assemble the guided table exactly as downstream (E3) expects it
rows = [{'support': k / N, 'itemsets': frozenset((i,)), 'k': 1}
        for i, k in c1.items() if i in amods]
rows += [{'support': k / N, 'itemsets': p, 'k': 2} for p, k in freq2.items()]
guided = (pd.DataFrame(rows)
            .sort_values('support', ascending=False)
            .reset_index(drop=True))
guided['has_amod'] = True
_n1 = int((guided.k == 1).sum()); _n2 = int((guided.k == 2).sum())
guided = guided.drop(columns='k')
guided.to_json(f'{ROOT}/results/frequent_itemsets.json', orient='records')
_lap(f'guided itemsets written: {len(guided):,}')

print(f'\nGuided itemsets (contain amod): {len(guided):,}  '
      f'[{_n1} singletons, {_n2} pairs]')
if len(guided) < 200:
    print('!! WARNING: still few guided patterns -- check transactions carry '
          'amod: items and consider lowering MIN_SUPPORT further.')
print('\nTop 12 adjective-bearing itemsets:')
for _, r in guided.head(12).iterrows():
    print(f"  {r.support:.4f}  {set(r.itemsets)}")
print('\nDiscriminative tail (lowest-support guided patterns):')
for _, r in guided.tail(12).iterrows():
    print(f"  {r.support:.5f}  {set(r.itemsets)}")

## Block E3 / Cell 4 — Elastic-net pattern weighting

**Model choice, documented honestly.** This notebook swaps the Bayesian Elastic
Net (PyMC) for its fast frequentist twin. Because the target is a geocell *class*
(not a continuous value), the correct frequentist form is multinomial logistic
regression with an elastic-net penalty (`penalty='elasticnet'`, `solver='saga'`)
— identical L1+L2 regularization geometry, so the feature-selection story carries
over unchanged. (The thesis result in Block I uses the full hybrid: frequentist
elastic net for bulk selection + PyMC on the selected top-k for credible intervals.)

**Feature matrix.** One binary column per single `amod:` pattern (multi-item
itemsets from Cell 3 characterize co-occurrence; singletons are what the
classifier consumes). Sparse CSR — the fit subsamples the ~798k training rows to
a stratified 60k (see code), which fits trivially in Colab RAM.

**Two outputs:** (1) the coefficient table — which adjective patterns carry
geographic signal, and (2) the per-sentence pattern feature vector consumed by
the fusion model in Block F2.


In [ ]:
from scipy import sparse
from sklearn.linear_model import LogisticRegression

# Vocabulary = amod patterns that survived support filtering, plus type: items
# (place-category nouns) so the model can weigh 'type:fort' vs 'amod:old_fort'.
amod_vocab = sorted({i for s in guided.itemsets for i in s
                     if i.startswith(('amod:', 'type:'))})

# --- DROP EXTRACTION ARTIFACTS ---------------------------------------------
# The dependency parser occasionally emits a lemma of '-' (dangling hyphen) or
# picks up Wikivoyage section headers ('Splurge', 'Sleep', ...) as adjectives /
# nouns. These are not linguistic patterns and pollute the Figure 2 table.
# Pattern body is '<adj>_<noun>' (amod:) or '<noun>' (type:); reject if any
# token is a bare hyphen, single char, non-alnum, or a known boilerplate word.
_STOP = {'splurge', 'budget', 'sleep', 'eat', 'drink', 'buy', 'connect',
         'see', 'do', 'go', 'get', 'stay', 'mid', 'man', 'woman', 'thing',
         'lot', 'one', 'way', 'bit'}
def _clean_pat(p):
    parts = p.split(':', 1)[1].split('_')
    return all(len(x) > 1 and x.isalnum() and x not in _STOP for x in parts)
_before = len(amod_vocab)
amod_vocab = sorted(p for p in amod_vocab if _clean_pat(p))
print(f'Pattern feature vocabulary: {len(amod_vocab)} '
      f'(dropped {_before - len(amod_vocab)} artifact/boilerplate patterns)')
# -------------------------------------------------------------------------

pat2col = {p: j for j, p in enumerate(amod_vocab)}

def featurize(tx_json_series):
    """Sentence transactions -> sparse binary pattern matrix (n x |vocab|)."""
    rows, cols = [], []
    for r, t in enumerate(tx_json_series):
        for item in set(json.loads(t)):
            j = pat2col.get(item)
            if j is not None:
                rows.append(r); cols.append(j)
    return sparse.csr_matrix((np.ones(len(rows), dtype=np.float32),
                              (rows, cols)),
                             shape=(len(tx_json_series), len(amod_vocab)))

X_all = featurize(annot.transaction)
tr = annot.split == 'train'

# sklearn >=1.8 deprecates penalty='elasticnet' (l1_ratio alone implies it);
# <1.8 REQUIRES it explicitly, and silently fits plain L2 if it is omitted.
# Version-gate rather than try/except: constructing an invalid combination
# does not raise until .fit(), so a try/except around the constructor can
# never select the fallback and would leave us silently fitting L2.
from sklearn import __version__ as _skv
_sk = tuple(int(x) for x in _skv.split('.')[:2])
_enet_kw = dict(solver='saga', l1_ratio=0.5, C=1.0, max_iter=150, tol=1e-3, n_jobs=-1)
if _sk < (1, 8):
    _enet_kw['penalty'] = 'elasticnet'
enet = LogisticRegression(**_enet_kw)

# --- SUBSAMPLE THE FIT --------------------------------------------------
# saga's cost scales with samples x features x classes. This stage only
# estimates ~|amod_vocab| weights per class for the geo-signal ranking (it is
# NOT the final classifier -- DistilBERT is), and those converge well below
# 100k samples. Fit on a stratified 60k subsample, keep the FULL matrix for
# Block F2 fusion features. max_iter=150 / tol=1e-3 keeps the fit under ~30 min
# at a few-hundred-class label space.
#
# REPORT THIS IN THE THESIS: pattern weights are fit on a stratified 60k
# subsample; FP-Growth and the fusion features use the full corpus.
ENET_FIT_MAX_N = 60_000

_tr_df = annot.loc[tr, ['label']]
if ENET_FIT_MAX_N is not None and len(_tr_df) > ENET_FIT_MAX_N:
    _frac = ENET_FIT_MAX_N / len(_tr_df)
    _fit_pos = (_tr_df.reset_index(drop=True)
                      .groupby('label', group_keys=False, observed=True)['label']
                      .apply(lambda g: g.sample(max(1, int(round(len(g) * _frac))),
                                                random_state=42))
                      .index.values)
    print(f'Elastic-net fit subsample: {len(_tr_df):,} -> {len(_fit_pos):,} rows '
          f'({_tr_df.label.nunique()} classes, all retained by stratification)')
else:
    _fit_pos = np.arange(len(_tr_df))
    print(f'Elastic-net fit on all {len(_fit_pos):,} training rows')

_X_tr = X_all[tr.values]
enet.fit(_X_tr[_fit_pos], _tr_df['label'].values[_fit_pos])

# Did saga actually converge, or just hit the iteration cap?
_niter = int(np.max(enet.n_iter_))
print(f'saga iterations: {_niter} / {_enet_kw["max_iter"]}'
      + ('  <-- HIT CAP, likely not converged; raise max_iter'
         if _niter >= _enet_kw['max_iter'] else '  (converged)'))

# Verify elastic net actually engaged (exact zeros are its hallmark); a silent
# L2 fallback would invalidate the "sparsifies vague adjectives" claim. But
# ~99%+ zeros means the fit COLLAPSED (too few samples/class), not elegant
# sparsity -- raise MIN_CELL in Cell 4 if that happens.
_n_zero = int((enet.coef_ == 0).sum())
_frac_zero = _n_zero / enet.coef_.size
print(f'Elastic net sanity: {_n_zero:,} / {enet.coef_.size:,} coefficients '
      f'exactly zero ({_frac_zero:.1%})'
      + ('  <-- WARNING: no zeros, check penalty!' if _n_zero == 0 else
         '  <-- WARNING: fit collapsed, raise MIN_CELL' if _frac_zero > 0.97 else ''))
# -----------------------------------------------------------------------

# Geo-signal score per pattern = max |coef| across classes: a pattern is
# informative if it strongly indicates ANY region.
signal = np.abs(enet.coef_).max(axis=0)
coef_table = (pd.DataFrame({'pattern': amod_vocab, 'geo_signal': signal})
                .sort_values('geo_signal', ascending=False))
coef_table.to_csv(f'{ROOT}/results/pattern_geo_signal.csv', index=False)

print('\nStrongest geo-signal patterns (paper Figure 2 preview):')
print(coef_table.head(15).to_string(index=False))
print('\nWeakest (the elastic net zeroing out vague adjectives, as designed):')
print(coef_table.tail(5).to_string(index=False))

# Persist the fusion features + vocab for Block F2
sparse.save_npz(f'{ROOT}/results/pattern_features.npz', X_all)
json.dump(amod_vocab, open(f'{ROOT}/results/pattern_vocab.json', 'w'))

## Cell 5 — Evaluation functions (defined once, used by every model)

Hits@k and nDCG@20 exactly as defined in Yousaf et al. (2023) §4.2, applied to
geocell rankings: for each test sentence we sort cells by predicted probability
and locate the rank of the true cell. `DCG = 1/log2(rank+1)`, ideal DCG = 1,
rank > 20 contributes 0 to nDCG@20 (their top-k convention).

In [ ]:
def rank_of_truth(prob_matrix, y_true):
    """For each row: 1-based rank of the true class when classes are sorted
    by descending predicted probability. Vectorized: rank = 1 + number of
    classes scored strictly higher than the true class."""
    true_scores = prob_matrix[np.arange(len(y_true)), y_true]
    return 1 + (prob_matrix > true_scores[:, None]).sum(axis=1)

def hits_ndcg(prob_matrix, y_true, ks=(1, 5, 10, 20)):
    ranks = rank_of_truth(np.asarray(prob_matrix), np.asarray(y_true))
    out = {f'Hits@{k}': float((ranks <= k).mean()) for k in ks}
    dcg = np.where(ranks <= 20, 1.0 / np.log2(ranks + 1), 0.0)
    out['nDCG@20'] = float(dcg.mean())
    out['MedianRank'] = float(np.median(ranks))
    return out

# self-test: perfect predictor -> all metrics 1.0
_p = np.eye(5)[[0, 1, 2]]
assert hits_ndcg(_p, [0, 1, 2])['Hits@1'] == 1.0
print('Eval functions ready and self-tested.')

## Cell 6 — Ablation A0: patterns-only classifier

How far do mined adjective patterns get you *without* any transformer? This is
the elastic-net model itself as a ranker — the cheapest row of the ablation
table and the sanity check that the mined features carry real signal (it should
beat random by a wide margin but lose clearly to DistilBERT).

In [ ]:
te_mask = (annot.split == 'test').values

# enet.predict_proba returns columns for enet.classes_ only (the classes present
# in the fit subsample) -- NOT the full label space. The OTHER class, and any
# train class the stratified subsample missed, are absent. Scatter into a
# full-width matrix so column j == global label id j; missing classes get prob 0
# and are ranked last (honest: the patterns-only model cannot predict them).
_proba = enet.predict_proba(X_all[te_mask])
probs_enet = np.zeros((_proba.shape[0], len(classes)), dtype=_proba.dtype)
probs_enet[:, enet.classes_] = _proba

res_A0 = hits_ndcg(probs_enet, annot.loc[te_mask, 'label'].values)
_rand = 1.0 / len(classes)
print(f'A0 patterns-only: ' + str({k: round(v, 4) for k, v in res_A0.items()}))
print(f'  (random baseline Hits@1 = {_rand:.4f}; A0 is {res_A0["Hits@1"] / _rand:.1f}x random)')

results_table = {'A0_patterns_only': res_A0}
json.dump(results_table, open(f'{ROOT}/results/ablation_results.json', 'w'), indent=2)
print('\nSWITCH TO T4 GPU NOW (Runtime -> Change runtime type), re-run Cell 1, continue at Cell 7.')

# Block F — DistilBERT (GPU from here)

## Cell 7 — Dataset + tokenizer

**Ordering per the runbook:** baseline FIRST (F1), fusion SECOND (F2) — so a
comparison exists even if the session dies at hour 20.

In [ ]:
import torch, json
import pandas as pd, numpy as np
from scipy import sparse
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset

assert torch.cuda.is_available(), 'No GPU — Block F genuinely needs the T4 (runbook Block F).'

annot = pd.read_parquet(f'{ROOT}/annotated/annotated_v1_labeled.parquet')
X_pat = sparse.load_npz(f'{ROOT}/results/pattern_features.npz')
n_classes = annot.label.nunique()
MODEL_NAME = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- TRAINING-SIZE CAP --------------------------------------------------
# One epoch over N examples at batch 32 costs N/32 steps; DistilBERT at
# seq-96 fp16 runs roughly 20-25 it/s on a T4. The full 799k train split is
# ~3-4h for 3 epochs per arm (F1 + F2 = 6-8h) with no headroom if the session
# drops. Capped to 300k (stratified, ~380/class, all 782 classes retained) ->
# ~1.5h per arm. F1 and F2 read the SAME mask so the comparison stays valid.
#
# IMPLEMENTED AS A BOOLEAN MASK, deliberately: Block F2 indexes X_pat by
# annot's index labels, which align positionally with X_pat's rows. Resampling
# or reindexing `annot` would silently desynchronise the pattern features from
# the text. Filtering by mask preserves those labels exactly.
#
# Set MAX_TRAIN_N = None to train on the full split.
# REPORTED IN THE THESIS: DistilBERT arms trained on a stratified 300k subsample.
MAX_TRAIN_N = 300_000

annot['use_for_training'] = annot.split == 'train'
if MAX_TRAIN_N is not None and int(annot.use_for_training.sum()) > MAX_TRAIN_N:
    _tr = annot.loc[annot.use_for_training, ['label']]
    _frac = MAX_TRAIN_N / len(_tr)
    _keep_idx = (_tr.groupby('label', group_keys=False, observed=True)['label']
                    .apply(lambda g: g.sample(max(1, int(round(len(g) * _frac))),
                                              random_state=42))
                    .index)
    annot['use_for_training'] = False
    annot.loc[_keep_idx, 'use_for_training'] = True
    print(f'Training cap: {len(_tr):,} -> {int(annot.use_for_training.sum()):,} rows '
          f'(stratified, all {_tr.label.nunique()} classes retained)')
else:
    print(f'Training on the full split: {int(annot.use_for_training.sum()):,} rows')
# -----------------------------------------------------------------------

def _split_mask(split):
    """Row mask for a split. 'train' additionally honours the size cap."""
    return annot.use_for_training if split == 'train' else (annot.split == split)

# In-training eval on a 25k stratified dev subsample -- full 101k dev every
# eval step would add ~40 min of pure eval overhead. Final F1/F2 metrics still
# use the full test set.
_dev_full = annot.index[annot.split == 'dev']
_dev_eval_idx = (annot.loc[_dev_full, ['label']]
                   .groupby('label', group_keys=False, observed=True)['label']
                   .apply(lambda g: g.sample(min(len(g), max(1, int(round(len(g) * 25_000 / len(_dev_full))))),
                                             random_state=42))
                   .index)
annot['use_for_dev_eval'] = False
annot.loc[_dev_eval_idx, 'use_for_dev_eval'] = True

def to_ds(split):
    if split == 'dev_eval':
        d = annot[annot.use_for_dev_eval]
    else:
        d = annot[_split_mask(split)]
    ds = Dataset.from_dict({'text': d.text.tolist(), 'labels': d.label.tolist()})
    return ds.map(lambda b: tok(b['text'], truncation=True, max_length=96),
                  batched=True, remove_columns=['text'])

ds_train, ds_dev, ds_test = to_ds('train'), to_ds('dev_eval'), to_ds('test')
print(f'train {len(ds_train)} | dev(eval) {len(ds_dev)} | test {len(ds_test)} | classes {n_classes}')

## Cell 8 — F1: baseline run (text-only)

This mirrors the *nouns-only spirit* of the 2023 paper: the transformer sees raw
text but receives no explicit adjective-pattern features. fp16, batch 32,
3 epochs ≈ 30–60 min on T4. Checkpoints every 500 steps to Drive;
`resume_from_checkpoint=True` makes a disconnect cost minutes.

In [ ]:
import glob as _glob, math
from transformers import EarlyStoppingCallback, DataCollatorWithPadding

model_b = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_classes)

# transformers 5.x removed warmup_ratio -> compute warmup_steps (6% of total)
_total_steps = math.ceil(len(ds_train) / 32) * 3

args_b = TrainingArguments(
    output_dir=f'{ROOT}/models/distilbert_baseline',
    per_device_train_batch_size=32, per_device_eval_batch_size=128,
    num_train_epochs=3, learning_rate=5e-5,
    warmup_steps=int(0.06 * _total_steps),
    weight_decay=0.01,
    fp16=True, eval_strategy='steps', eval_steps=500,
    save_strategy='steps', save_steps=500, save_total_limit=2,
    # Keep the checkpoint with the lowest dev loss, not the last one -- so a
    # late-epoch overfit does not become the model we evaluate and ship.
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=100, report_to='none', seed=42)

# transformers 5.x no longer auto-infers a padding collator -- the default one
# stacks raw lists and dies on variable-length sequences. Pad per batch.
trainer_b = Trainer(model=model_b, args=args_b,
                    train_dataset=ds_train, eval_dataset=ds_dev,
                    data_collator=DataCollatorWithPadding(tokenizer=tok),
                    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

# Resume only from a real checkpoint-* dir (a bare output dir with just a
# config must NOT trigger resume -- and the training schedule is tied to the
# 300k cap, so a checkpoint from a different train size would resume onto the
# wrong LR schedule).
_ckpts = _glob.glob(f'{args_b.output_dir}/checkpoint-*')
trainer_b.train(resume_from_checkpoint=bool(_ckpts))

_pred = trainer_b.predict(ds_test).predictions
_pred = _pred[0] if isinstance(_pred, tuple) else _pred
probs_b = torch.softmax(torch.tensor(_pred), -1).numpy()
res_F1 = hits_ndcg(probs_b, annot.loc[annot.split == 'test', 'label'].values)
print('F1 DistilBERT baseline:', {k: round(v, 4) for k, v in res_F1.items()})
results_table = json.load(open(f'{ROOT}/results/ablation_results.json'))
results_table['F1_distilbert_baseline'] = res_F1
json.dump(results_table, open(f'{ROOT}/results/ablation_results.json', 'w'), indent=2)
np.save(f'{ROOT}/results/probs_baseline.npy', probs_b)

## Cell 9 — F2: fusion model (text ⊕ mined pattern features)

**Architecture.** DistilBERT's pooled [CLS] representation (768-d) is
concatenated with the elastic-net pattern vector projected to 64-d, then a
2-layer MLP head classifies. If this row of the ablation table beats F1, the
mined adjective patterns add information the transformer did not extract from
raw text on its own.

The custom `collate` carries each sentence's pattern vector alongside its tokens.
F2 uses the same train/dev/test mask as F1 (`_split_mask`), but note the training
schedules differ: F1 (Cell 8) uses `Trainer` with early stopping and
best-by-dev-loss checkpointing, while F2 below runs a fixed 3 epochs, last epoch.
Report this asymmetry with the F2 − F1 delta.


In [ ]:
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import DataLoader

class FusionGeoTagger(nn.Module):
    """DistilBERT [CLS] (768) ++ projected pattern features (64) -> geocell."""
    def __init__(self, model_name, n_patterns, n_classes, proj_dim=64):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.proj = nn.Sequential(nn.Linear(n_patterns, proj_dim), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(768 + proj_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, n_classes))
    def forward(self, input_ids, attention_mask, patterns):
        cls = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.head(torch.cat([cls, self.proj(patterns)], dim=-1))

def make_loader(split, shuffle):
    # Same mask as Block F1 so the baseline and fusion arms are trained on an
    # identical row set -- otherwise the F1-vs-F2 comparison is confounded.
    d = annot[_split_mask(split)]
    idx = d.index.values          # index labels align with X_pat rows
    enc = tok(d.text.tolist(), truncation=True, max_length=96,
              padding=False)
    items = [{'input_ids': enc['input_ids'][i],
              'attention_mask': enc['attention_mask'][i],
              'row': idx[i], 'label': d.label.iloc[i]} for i in range(len(d))]
    def collate(batch):
        maxlen = max(len(b['input_ids']) for b in batch)
        ids  = torch.zeros(len(batch), maxlen, dtype=torch.long)
        mask = torch.zeros(len(batch), maxlen, dtype=torch.long)
        for i, b in enumerate(batch):
            L = len(b['input_ids'])
            ids[i, :L]  = torch.tensor(b['input_ids'])
            mask[i, :L] = torch.tensor(b['attention_mask'])
        pats = torch.tensor(
            X_pat[[b['row'] for b in batch]].toarray(), dtype=torch.float32)
        labels = torch.tensor([b['label'] for b in batch])
        return ids, mask, pats, labels
    return DataLoader(items, batch_size=32 if shuffle else 128,
                      shuffle=shuffle, collate_fn=collate)

# --- PATTERN COVERAGE GATE --------------------------------------------
# The fusion branch can only add information on sentences that actually
# carry >= 1 mined pattern. If coverage is near zero, F2 cannot beat F1 by
# construction and any F2 - F1 delta is noise. Check before spending a GPU
# session; if train coverage < 5%, lower MIN_SUPPORT in E2 and re-run E2/E3.
_cov = {}
for _s in ('train', 'test'):
    _rows = annot[_split_mask(_s)].index.values
    _active = np.asarray((X_pat[_rows] != 0).sum(axis=1)).ravel()
    _cov[_s] = (_active > 0).mean()
    print(f'{_s}: pattern coverage {_cov[_s]:.1%} '
          f'({int((_active > 0).sum()):,}/{len(_rows):,}), '
          f'mean active patterns/sentence {_active.mean():.2f}')
assert _cov['train'] > 0.05, (
    f"Train pattern coverage {_cov['train']:.1%} < 5% -- F2 cannot demonstrate "
    "a contribution. Lower MIN_SUPPORT in E2 and re-run E2/E3 before training.")
# ---------------------------------------------------------------------

dl_train, dl_test = make_loader('train', True), make_loader('test', False)

dev = 'cuda'
fusion = FusionGeoTagger(MODEL_NAME, X_pat.shape[1], n_classes).to(dev)
opt = torch.optim.AdamW(fusion.parameters(), lr=5e-5)
# torch>=2.4 moved GradScaler to torch.amp; the old path warns and is slated
# for removal. Prefer the new API, fall back for older Colab images.
try:
    scaler = torch.amp.GradScaler('cuda')
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler()
lossf = nn.CrossEntropyLoss()
CKPT = f'{ROOT}/models/fusion_ckpt.pt'
start_ep = 0
if os.path.exists(CKPT):                      # disconnect resume
    # weights_only=False: the checkpoint carries optimizer state (non-tensor
    # objects); torch>=2.6 defaults weights_only=True and would refuse to load.
    st = torch.load(CKPT, map_location=dev, weights_only=False)
    fusion.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
    start_ep = st['epoch'] + 1
    print(f'Resumed from epoch {start_ep}')

for ep in range(start_ep, 3):
    fusion.train(); tot = 0.0
    for ids, mask, pats, labels in dl_train:
        ids, mask, pats, labels = ids.to(dev), mask.to(dev), pats.to(dev), labels.to(dev)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            loss = lossf(fusion(ids, mask, pats), labels)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item()
    torch.save({'model': fusion.state_dict(), 'opt': opt.state_dict(),
                'epoch': ep}, CKPT)          # checkpoint after EVERY epoch
    print(f'epoch {ep}: mean loss {tot/len(dl_train):.4f} (checkpointed)')

fusion.eval(); all_p = []
with torch.no_grad():
    for ids, mask, pats, _ in dl_test:
        with torch.cuda.amp.autocast():
            logits = fusion(ids.to(dev), mask.to(dev), pats.to(dev))
        all_p.append(torch.softmax(logits.float(), -1).cpu().numpy())
probs_f = np.vstack(all_p)
np.save(f'{ROOT}/results/probs_fusion.npy', probs_f)   # persist before the metric call
res_F2 = hits_ndcg(probs_f, annot.loc[annot.split == 'test', 'label'].values)
print('F2 fusion:', {k: round(v, 4) for k, v in res_F2.items()})
results_table = json.load(open(f'{ROOT}/results/ablation_results.json'))
results_table['F2_fusion'] = res_F2
json.dump(results_table, open(f'{ROOT}/results/ablation_results.json', 'w'), indent=2)

## Block G / Cell 10 — The ablation table

The single most important artifact of the sprint. Each row answers one question:
- **A0**: do mined adjective patterns alone carry geo-signal?
- **F1**: what does a text-only transformer achieve (the nouns-era ceiling)?
- **F2 − F1**: *the contribution* — the marginal value of adjective patterns.

In [ ]:
results_table = json.load(open(f'{ROOT}/results/ablation_results.json'))
tbl = pd.DataFrame(results_table).T[['Hits@1','Hits@5','Hits@10','Hits@20','nDCG@20','MedianRank']]
print(tbl.round(4).to_string())
tbl.to_csv(f'{ROOT}/results/ablation_table.csv')

delta = tbl.loc['F2_fusion'] - tbl.loc['F1_distilbert_baseline']
print('\nContribution of adjective patterns (F2 - F1):')
print(delta.round(4).to_string())

## Cell 11 — G2: external evaluation on the authors' paraphrased dataset

> **Superseded.** This cell is an early, weak version of the base-paper comparison: it uses
> spaCy `en_core_web_md` (GloVe) vectors — the *worst* embedding in Yousaf et al. (2023), nDCG
> 0.19 — and adds all adjectives indiscriminately, which dilutes. The proper reproduction (their
> Word2Vec pipeline, Algorithms 2 **and** 3, faithful metrics) and the adjective extension are in
> **Sprint Block H**; the FP-Growth + Bayesian Elastic Net selection is in **Sprint Block I**.
> Kept here only as the record of the first attempt.

**What is and isn't comparable.** Their task ranks OSM tags for paraphrased places; the rest of
*this* notebook ranks geocells, so the DistilBERT model is not comparable to their Table 4. Only
their semantic-similarity protocol is: rank OSM tags for their sentences, nouns-only (their
setting) vs with our adjectives added — one variable changed. Their published context nouns and
complete tag list are loaded straight from the repo (106 aligned rows).


In [ ]:
%%capture
!pip install -q spacy && python -m spacy download en_core_web_md -q

In [ ]:
import requests, sys, re
sys.path.insert(0, ROOT)
import spacy
import adjective_extractor as ax

# --- 1. fetch the published dataset (actual repo layout) -----------------
# dwolter/place-descriptions stores the paraphrased eval set as THREE files, not
# a single CSV:
#   descriptions.txt              -- one sentence/line, spatial nouns wrapped as
#                                    '(:spatial [:primary] <words>)'
#   paraphrased/full_input_tokens.txt -- alternating lines: context nouns, then
#                                        the gold OSM tag
#   paraphrased/osm_tags_complete.txt -- the full candidate-tag list (their setting)
BASE = 'https://raw.githubusercontent.com/dwolter/place-descriptions/main/'

def _strip_markup(l):
    l = re.sub(r'\\footnote\{[^}]*\}?', ' ', l)      # one line carries a LaTeX footnote
    l = l.replace('(', ' ').replace(')', ' ').replace(',', ' ')
    l = l.replace(':spatial', ' ').replace(':primary', ' ')
    return re.sub(r'\s+', ' ', l).strip()

sentences = [_strip_markup(l) for l in requests.get(BASE + 'descriptions.txt').text.splitlines() if l.strip()]

_tok = [l.strip() for l in requests.get(BASE + 'paraphrased/full_input_tokens.txt').text.splitlines() if l.strip()]
their_nouns = [[w.strip() for w in _tok[i].split(',') if w.strip()] for i in range(0, len(_tok), 2)]
gold_tags   = [_tok[i].lower().replace(' ', '_')                     for i in range(1, len(_tok), 2)]

assert len(sentences) == len(gold_tags) == len(their_nouns), \
    f'misaligned: {len(sentences)} sentences / {len(gold_tags)} tags / {len(their_nouns)} noun-rows'
ext = pd.DataFrame({'sentence': sentences, 'gold_tag': gold_tags, 'their_nouns': their_nouns})
print(f'External eval sentences: {len(ext)}')

# --- 2. candidate OSM tag lexicon = their published complete list --------
OSM_TAGS = sorted({t.strip().lower().replace(' ', '_')
                   for t in requests.get(BASE + 'paraphrased/osm_tags_complete.txt').text.splitlines()
                   if t.strip()})
print(f'Candidate OSM tags: {len(OSM_TAGS)}  '
      f'(gold tags in lexicon: {sum(g in OSM_TAGS for g in ext.gold_tag)}/{len(ext)})')

nlp_md = spacy.load('en_core_web_md')
pi = ax.PlaceInterpreter(model='en_core_web_md')
tag_mat = np.vstack([nlp_md(t.replace('_', ' ')).vector for t in OSM_TAGS])
tag_mat /= (np.linalg.norm(tag_mat, axis=1, keepdims=True) + 1e-9)

def _score(words):
    """Summed [0,1]-rescaled cosine similarity of each context word to every tag
    (their Algorithm 2)."""
    score = np.zeros(len(OSM_TAGS))
    for w in words:
        v = nlp_md(w).vector
        n = np.linalg.norm(v)
        if n < 1e-6:
            continue
        sims = tag_mat @ (v / n)
        rng = sims.max() - sims.min()
        score += (sims - sims.min()) / (rng + 1e-9)
    return score

def rank_tags(row, use_adjectives):
    """Nouns-only reproduces their setting using their published context nouns.
    The adjective arm adds our extractor's adjectives as extra context words plus
    DIRECT/MODIFIER tag-hint boosts -- the one variable changed."""
    words = list(row.their_nouns)
    hint_tags = []
    if use_adjectives:
        r = pi.interpret(row.sentence)
        for e in r.entities:
            words += e['adjectives']
        for h in r.tag_hints:
            v = h['value'] if h['value'] != '*' else h['key']
            if v in OSM_TAGS:
                hint_tags.append(v)
    if not words and not hint_tags:
        return OSM_TAGS[:]
    ranked = [OSM_TAGS[i] for i in np.argsort(-_score(words))]
    for t in reversed(hint_tags):
        if t in ranked:
            ranked.remove(t); ranked.insert(0, t)
    return ranked

def eval_external(use_adjectives):
    ranks = []
    for _, row in ext.iterrows():
        ranked = rank_tags(row, use_adjectives)
        ranks.append(ranked.index(row.gold_tag) + 1 if row.gold_tag in ranked
                     else len(ranked) + 1)
    ranks = np.array(ranks)
    return {f'Hits@{k}': round(float((ranks <= k).mean()), 4) for k in (1, 5, 10, 20)} | \
           {'nDCG@20': round(float(np.where(ranks <= 20, 1 / np.log2(ranks + 1), 0).mean()), 4)}

res_nouns = eval_external(use_adjectives=False)
res_adj   = eval_external(use_adjectives=True)
print('Nouns-only (reproducing their setting):', res_nouns)
print('With adjectives (ours):               ', res_adj)
print('Their published best (W2V clustered):  Hits@1=0.2264, Hits@5=0.5283, '
      'Hits@10=0.6509, Hits@20=0.7075, nDCG@20=0.4442')
json.dump({'external_nouns_only': res_nouns, 'external_with_adjectives': res_adj},
          open(f'{ROOT}/results/external_eval.json', 'w'), indent=2)

## Cell 12 — Final results bundle

Everything the paper needs, in one folder: ablation table, pattern geo-signal
coefficients, frequent itemsets, external comparison, and per-sentence test
predictions for error analysis.

In [ ]:
# per-sentence predictions for the manual error-analysis pass (runbook G, last step)
test_d = annot[annot.split == 'test'].copy()
test_d['pred_rank_baseline'] = rank_of_truth(np.load(f'{ROOT}/results/probs_baseline.npy'),
                                             test_d.label.values)
# Load F2 predictions from disk (Cell 9 persists them) so this cell survives a
# runtime restart between F2 training and the bundle step.
test_d['pred_rank_fusion'] = rank_of_truth(np.load(f'{ROOT}/results/probs_fusion.npy'),
                                           test_d.label.values)
test_d[['sent_id','text','source','geocell','pred_rank_baseline','pred_rank_fusion']]\
    .to_csv(f'{ROOT}/results/test_predictions.csv', index=False)

print('results/ contents:')
for f in sorted(glob.glob(f'{ROOT}/results/*')):
    print('  ', f.split('/')[-1])
print('\nSPRINT COMPLETE. Hours 24-48 = buffer / error analysis / paper tables.')